# NOTEBOOK 04 – TRÚC CHI | CHƯƠNG 4: MÔ HÌNH & THỰC NGHIỆM

**Phụ trách:** 4.1–4.5 · Issue **#5** · Hạn báo cáo/Validation: 22/11/2026; bàn giao dự đoán Test cho Vĩnh Chi trước 24/11.

**Đầu vào** từ Lam Linh: `data/processed/orders_model_ready.csv`. **Đầu ra**: `test_predictions.csv`, `experiment_metadata.json`, các model `joblib` (không push), bảng tham số và ngưỡng trên Validation.

[Lý thuyết Ch4](../docs/theory/04_truc_chi_ch4.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

> Chạy **chỉ sau khi Ch3 bàn giao** và audit. Không đưa thông tin tương lai vào X. Không báo kết quả trước khi chạy; các cell ở đây là code gợi ý chuẩn hóa để hoàn thiện.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 4.1 Chia Train/Validation/Test theo thời gian

Trước khi chia:
- Dữ liệu có 1 dòng/`order_id`, nhãn `is_late` hợp lệ và thời điểm `order_approved_at`.
- Giữ `order_delivered_customer_date` **ngoài feature X**, chỉ dùng kiểm tra nhãn đã xuất hiện trước cutoff.
- Chia thử 70/15/15 theo thời gian duyệt; đối chiếu số mẫu và tỷ lệ nhãn từng tập.
- Để mô phỏng dự báo theo thời gian, nhãn Train phải chín muồi trước cutoff Validation; nhãn Validation phải chín muồi trước cutoff Test. Ghi số bị loại.

In [ ]:
from src.contracts import assert_unique_orders
source = PROCESSED / "orders_model_ready.csv"
if not source.exists():
    raise FileNotFoundError("Thiếu data/processed/orders_model_ready.csv. Nhờ Lam Linh chạy notebook 03 và bàn giao.")
df = pd.read_csv(source, parse_dates=["order_approved_at", "order_delivered_customer_date"])
required = {"order_id", "is_late", "order_approved_at", "order_delivered_customer_date"}
assert required.issubset(df.columns), required.difference(df.columns)
assert_unique_orders(df)
df = df.loc[df["order_approved_at"].notna() & df["is_late"].isin([0,1])].copy()
df = df.sort_values(["order_approved_at", "order_id"]).reset_index(drop=True)
n = len(df)
assert n >= 30, "Quá ít đơn để chia 3 tập; kiểm tra tiền xử lý."
a, b = int(n * 0.70), int(n * 0.85)
raw_train, raw_val, test = df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()
val_start, test_start = raw_val["order_approved_at"].min(), test["order_approved_at"].min()
train = raw_train.loc[raw_train["order_delivered_customer_date"] < val_start].copy()
val = raw_val.loc[raw_val["order_delivered_customer_date"] < test_start].copy()
assert len(train) and len(val) and len(test), "Không đủ nhãn đã chín muồi tại cutoff."
assert train["order_approved_at"].max() <= val["order_approved_at"].min()
assert val["order_approved_at"].max() <= test["order_approved_at"].min()
assert set(train.order_id).isdisjoint(val.order_id)
assert set(train.order_id).isdisjoint(test.order_id)
assert set(val.order_id).isdisjoint(test.order_id)
display(pd.DataFrame([
    {"split":name, "n":len(x), "late_rate": x.is_late.mean()}
    for name,x in [("train",train),("validation",val),("test",test)]
]))
print("Bị loại khỏi Train do nhãn chưa có ở validation cutoff:", len(raw_train)-len(train))
print("Bị loại khỏi Validation do nhãn chưa có ở test cutoff:", len(raw_val)-len(val))

## 4.2 Tiền xử lý thống nhất và Dummy baseline

**Feature whitelist an toàn tối thiểu:** tháng, thứ/giờ đặt hàng, khoảng cách từ đặt tới duyệt và bang khách. Chỉ mở rộng bằng giá/phí/seller/product/estimated date sau khi xác minh hợp đồng dữ liệu.

Dùng `ColumnTransformer`, `SimpleImputer`, `OneHotEncoder`; MLP cần `StandardScaler` cho numeric. Mỗi pipeline tự fit trên Train, Validation/Test chỉ transform.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
import joblib, json

NUMERIC = ["purchase_month_number", "purchase_weekday", "purchase_hour", "approval_delay_hours"]
CATEGORICAL = ["customer_state"]
FEATURES = NUMERIC + CATEGORICAL
assert set(FEATURES).issubset(df.columns), set(FEATURES).difference(df.columns)
X_train, y_train = train[FEATURES], train["is_late"].astype(int)
X_val, y_val = val[FEATURES], val["is_late"].astype(int)
X_test, y_test = test[FEATURES], test["is_late"].astype(int)

def make_preprocessor(scale_numeric=False):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), NUMERIC),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORICAL),
    ])

models = {
    "dummy": Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", DummyClassifier(strategy="prior"))
    ]),
    "rf": Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                         class_weight="balanced_subsample", n_jobs=-1,
                                         random_state=42))
    ]),
    "mlp": Pipeline([
        ("prep", make_preprocessor(True)),
        ("model", MLPClassifier(hidden_layer_sizes=(64,32), alpha=0.001,
                                early_stopping=True, max_iter=180, random_state=42))
    ]),
}
assert y_train.nunique() == 2, "Train thiếu 1 lớp: kiểm tra lại split."

## 4.3–4.4 Huấn luyện Random Forest và MLP

**Bài cần trình bày:** cài đặt và tham số, căn cứ chọn, thời gian huấn luyện, kết quả Validation, biểu đồ loss MLP nếu có. Giải thích vì sao mô hình thắng/thua, không ưu tiên Deep Learning theo định kiến.

Cell dưới huấn luyện cả 3 mô hình (Dummy, RF, MLP) trên cùng Train và đánh giá **chỉ Validation** để chọn mô hình/ngưỡng.

In [ ]:
val_prob = {}
val_results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    val_prob[name] = proba
    val_results.append({"model": name, "val_AP": average_precision_score(y_val, proba)})
val_scores = pd.DataFrame(val_results).sort_values("val_AP", ascending=False)
display(val_scores)
print("Chưa dùng tập Test để lựa chọn tham số/threshold.")

## 4.5 Chọn ngưỡng cảnh báo và xếp hạng Top-K trên Validation

Duyệt threshold với từng model và chọn F1 cao nhất *trên Validation* (quy ước khởi đầu; có thể đổi sang mục tiêu Recall/chi phí FP nếu nhóm chốt trước khi xem Test).
Mô hình chính chọn theo AP Validation giữa **Random Forest** và **MLP**; Dummy là tham chiếu.

**Không điều chỉnh threshold bằng kết quả Test.** Ghi rõ tỷ lệ lớp dương và ngân sách Top-K theo nhu cầu vận hành.

In [ ]:
from sklearn.metrics import confusion_matrix
threshold_grid = np.linspace(0.10, 0.90, 17)
thresholds = {}
threshold_report = []
for name, probs in val_prob.items():
    candidates = []
    for threshold in threshold_grid:
        pred = (probs >= threshold).astype(int)
        candidates.append({
            "threshold": float(threshold),
            "precision": precision_score(y_val, pred, zero_division=0),
            "recall": recall_score(y_val, pred, zero_division=0),
            "f1": f1_score(y_val, pred, zero_division=0),
            "false_alerts": int(((pred == 1) & (y_val.to_numpy() == 0)).sum())
        })
    best = sorted(candidates, key=lambda q: (-q["f1"], q["false_alerts"]))[0]
    thresholds[name] = best["threshold"]
    threshold_report.append({"model":name, **best})
display(pd.DataFrame(threshold_report))
candidates = [x for x in val_results if x["model"] in {"rf","mlp"}]
champion = max(candidates, key=lambda x:x["val_AP"])["model"]
chosen_threshold = thresholds[champion]
print("Champion theo Validation:", champion, "| threshold:", chosen_threshold)

## 4.6 Đóng băng kết quả Test và bàn giao Chương 5

Chỉ chạy cell dưới **một lần sau khi chốt** cấu hình/mốc/ngưỡng trên Validation. Lưu `test_predictions.csv` chứa `order_id`, `y_true`, `p_dummy`, `p_rf`, `p_mlp`, `y_pred_champion`; Ch5 sử dụng trực tiếp.

Tất cả file CSV/model dưới `data/processed/` và `models/` đã được .gitignore; ghi cách tái lập thay vì push dữ liệu vào repository.

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
score_test = {name: model.predict_proba(X_test)[:, 1] for name, model in models.items()}
predictions = pd.DataFrame({
    "order_id": test["order_id"].to_numpy(),
    "y_true": y_test.to_numpy(),
    "p_dummy": score_test["dummy"],
    "p_rf": score_test["rf"],
    "p_mlp": score_test["mlp"],
})
predictions["y_pred_champion"] = (
    predictions["p_" + champion].to_numpy() >= chosen_threshold
).astype(int)
assert len(predictions) == len(test)
assert predictions["order_id"].is_unique
predictions.to_csv(PROCESSED / "test_predictions.csv", index=False)
joblib.dump(models[champion], MODEL_DIR / "champion.joblib")
meta = {
    "prediction_time": "order_approved_at",
    "features": FEATURES,
    "champion": champion,
    "thresholds_from_val": thresholds,
    "chosen_threshold": chosen_threshold,
    "validation_AP": {x["model"]: float(x["val_AP"]) for x in val_results},
    "split_counts": {"train":len(train), "val":len(val), "test":len(test)},
    "cutoffs": {"validation_start": str(val_start), "test_start": str(test_start)},
    "label": "DATE(actual)>DATE(estimated) on labeled delivered orders",
    "random_state": 42,
}
(PROCESSED / "experiment_metadata.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu test_predictions, champion.joblib, experiment_metadata.json")
print("Đầu ra cần bàn giao cho Vĩnh Chi (Ch5) và Tấn Lên (Ch6).")

## 4.7 Checklist báo cáo & bàn giao
- [ ] Temporal split không trùng order_id, kiểm soát thời điểm trưởng thành nhãn.
- [ ] Có bảng train/val/test, tỷ lệ trễ từng tập, cutoff và số loại do chưa đủ nhãn.
- [ ] Preprocessing chỉ fit Train; cùng split cho RF/MLP/Dummy.
- [ ] Có log cấu hình, kết quả Validation, ngưỡng chốt.
- [ ] File Test predictions và champion model có đủ metadata.
- [ ] Có lời giải thích tránh Data Leakage và giới hạn delivered-only.
- [ ] Ch5 đã xác nhận nhận file; đã hoàn thành `reports/chapters/CH04_MODELING.md`.

**Hướng dẫn trình bày:** mô tả setup → phương pháp → tham số → Validation → tiêu chí chọn mô hình → artifact bàn giao; không công bố kết quả Test làm căn cứ chọn model.